# JAX-MD (jaxmd20) runtime

Average runtime of the reverse-mode gradient for the `jaxmd20` benchmark, per pass pipeline and AD placement.
Recreates `jaxmd20-runtime.pdf` from `results_JAXMD.csv`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# works whether the notebook is run from the repo root or from `results/`
csv = Path("results_JAXMD.csv")
if not csv.exists():
    csv = Path("..") / csv

df = pd.read_csv(csv)
df

In [ ]:
# CSV key -> group label
KEYS = {
    "PreRev": "Pre AD",
    "PostRev": "Post AD",
    "BothRev": "Pre + Post AD",
}

# CSV pass pipeline -> legend label
PIPELINES = {
    "HLOOpt": "HLO Opts.",
    "DefOpt": "All (No Inline)",
    "IDefOpt": "All (Inline)",
}

COLORS = {
    "HLOOpt": "#fc8d62",
    "DefOpt": "#66c2a5",
    "IDefOpt": "#8da0cb",
}

avg = df.pivot_table(index="Key", columns="Pass Pipeline", values="Avg Time")  # seconds
baseline = avg.loc["BothRev", "Jax"]  # "No opt"

avg.loc[list(KEYS), list(PIPELINES)].assign(**{"No opt (Jax)": baseline})

## Runtime

In [ ]:
width = 0.25
bar_kw = dict(edgecolor="black", linewidth=0.8)

fig, ax = plt.subplots(figsize=(7, 3.5))

# "No opt" group: single wide bar
ax.bar(0, baseline, width=len(PIPELINES) * width, color="lightgray", label="No opt", **bar_kw)

# Pre AD / Post AD / Pre + Post AD groups
x = np.arange(1, len(KEYS) + 1)
offsets = (np.arange(len(PIPELINES)) - (len(PIPELINES) - 1) / 2) * width
for off, (pipe, label) in zip(offsets, PIPELINES.items()):
    ax.bar(x + off, avg.loc[list(KEYS), pipe], width, color=COLORS[pipe], label=label, **bar_kw)

ax.set_xticks(np.arange(len(KEYS) + 1), ["No opt", *KEYS.values()])
ax.set_ylabel("Runtime [s]")
ax.legend(frameon=False)
fig.tight_layout()

# fig.savefig("jaxmd20-runtime.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# speedup w.r.t. "No opt" (Jax, BothRev): > 1 means faster than the baseline
speedup = (baseline / avg.loc[list(KEYS), list(PIPELINES)]).rename(index=KEYS, columns=PIPELINES)
speedup.round(2)

## Speedup over JAX

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))

x = np.arange(len(KEYS))
for off, (pipe, label) in zip(offsets, PIPELINES.items()):
    ax.bar(x + off, speedup[label], width, color=COLORS[pipe], label=label, **bar_kw)

ax.axhline(1, color="black", linestyle="--", linewidth=0.8)  # "No opt" baseline
ax.set_xticks(x, list(KEYS.values()))
ax.set_ylabel('Speedup over "No opt"')
ax.legend(frameon=False, ncols=len(PIPELINES), loc="lower center", bbox_to_anchor=(0.5, 1.0))
fig.tight_layout()

# fig.savefig("jaxmd20-speedup.pdf", bbox_inches="tight")
plt.show()